In [47]:
from typing import TypedDict

from langgraph.graph import StateGraph, START,END

from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint

from langchain_core.prompts import ChatPromptTemplate

from dotenv import load_dotenv
import os


In [48]:
load_dotenv()

True

In [49]:
endpoint = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN"),
    temperature=0.3,
    max_new_tokens=300,
)
llm = ChatHuggingFace(llm=endpoint)

In [50]:
class WorkflowState(TypedDict):
    text: str

In [51]:
prompt = ChatPromptTemplate.from_template(
"""
You are an expert editor.

Improve the following paragraph.

Paragraph:
{text}
"""
)

chain = prompt | llm

In [52]:
def improve_text(state: WorkflowState):

    response = chain.invoke(
        {
            "text": state["text"]
        }
    )

    return {
        "text": response
    }

In [53]:
graph = StateGraph(WorkflowState)

In [54]:
graph.add_node(
    "improve_text",
    improve_text
)

In [55]:
graph.add_edge(
    START,
    "improve_text"
)

In [56]:
graph.add_edge(
    "improve_text",
    END
)

In [57]:
app = graph.compile()

In [58]:
result = app.invoke(
    {
        "text":
        """
        hello sir,
        i wants internship in your company because i likes AI.
        """
    }
)

In [59]:
print(result["text"])

content="Certainly! Here's an improved version of the paragraph:\n\n---\n\nDear Sir,\n\nI am writing to express my interest in securing an internship at your company. I have a strong passion for AI and believe that this internship would provide me with valuable experience and insights into the field.\n\n---\n\nThis version is more professional and clearly conveys your interest and motivation." additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 70, 'prompt_tokens': 60, 'total_tokens': 130}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f7a9f-3cad-7b72-b424-ce1f11651c21-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 60, 'output_tokens': 70, 'total_tokens': 130}
